In [1]:
import pandas as pd
import json
import sys
from pathlib import Path

# Project root
PROJECT_ROOT = Path.cwd().parent
# Add project root to Python path
sys.path.insert(0, str(PROJECT_ROOT))

from src.llm.models import FinancialLLM

llm = FinancialLLM()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [2]:
prompt = """
You are a financial assistant.

Answer the following question:

How has Apple's revenue grown?
"""

response = llm.generate(prompt)

print(response)

Apple's revenue grew significantly over time due to several key factors:

1. Innovation: Apple continues to invest heavily in innovation and technology development, which leads to increased sales of its products. This includes new iPhone models, MacBook computers, and other tech gadgets.

2. Product diversification: Apple has expanded its product range beyond just smartphones and laptops into various categories like MacBooks, iPads, HomePod speakers, and Apple Watch.

3. Expansion into emerging markets: The company has aggressively entered new markets such as China, India, South Korea, and Japan, expanding its global presence and increasing demand for its products.

4. Global expansion: Apple has made significant investments in international markets, including Brazil, Russia, and Southeast Asia, which helps to spread its brand and increase customer base.

5. Brand loyalty: Apple's strong brand image and commitment to quality have helped to build customer loyalty and drive repeat busine

In [3]:
prompt = "Explain what revenue means in simple terms."

response = llm.generate(prompt)

print(response)

Revenue is the amount of money that an organization earns from its business activities or operations. It represents the total income generated over a specific period of time, such as a year, quarter, month, or day.

In simpler terms, revenue can be thought of as the financial gain an entity receives from selling goods and services to customers, generating income for its owners or shareholders. Revenue includes both cash inflows (such as sales of products or services) and non-cash inflows (like investments or dividends received).

Understanding revenue helps businesses track their performance, make informed decisions about pricing strategies, and identify areas where they might need to adjust their business models or marketing efforts to increase profitability.


In [4]:
prompt = "What is the difference between revenue and net income?"

response = llm.generate(prompt)

print(response)

Revenue refers to the total amount of money that an organization generates from its operations, including sales of goods or services, provision of services, or other business activities. Revenue can be measured in various ways such as direct sales, indirect sales, or customer transactions.

Net Income, on the other hand, is calculated by subtracting all expenses (such as salaries, rent, utilities, marketing, etc.) from the total revenue generated by an organization. Net income represents the profitability of an organization after accounting for all costs associated with its operations.

In summary, revenue measures the total financial benefit gained from selling products or providing services, while net income reflects the overall profitability of an organization. Both metrics help organizations evaluate their financial performance and make informed decisions about their future strategies.


In [5]:
prompt = "What is the difference between revenue and net income?"

response = llm.generate(prompt)

print(response)

Revenue refers to the total amount of money that a company generates from its operations or sales activities. It represents the value of products or services sold to customers and the expenses incurred in producing those products or providing services.

Net Income, on the other hand, is the difference between a company's revenues and its expenses over a specific period. It includes all profit earned by the company after deducting all costs (including taxes) from its revenues. Net income can be used as an indicator of a company's profitability and financial health.

In summary, revenue measures the overall economic activity generated by a company, while net income reflects the actual profits earned by the company after accounting for all expenses.


###  Create a tool routing prompt

In [6]:
from src.llm.tool_definitions import TOOLS

In [7]:
def build_tool_prompt(question):
    
    tool_descriptions = []

    for name, tool in TOOLS.items():
        tool_descriptions.append(
            f"""
Tool: {name}

Description:
{tool["description"]}

Input:
{tool["input"]}

Output:
{tool["output"]}
"""
        )

    tools_text = "\n".join(tool_descriptions)

    prompt = f"""
You are a financial analysis assistant.

Your task is to select the single best tool
to answer the user's question.

Available tools:

{tools_text}

User question:
{question}

Return your answer in the following JSON format:

{{
    "tool": "tool_name",
    "arguments": {{
        "company": "company_name"
    }}
}}

Return ONLY the JSON object.
"""

    return prompt

In [8]:
question = "How has Apple's revenue grown?"

prompt = build_tool_prompt(question)

response = llm.generate(prompt)

print(response)

{
    "tool": "get_revenue_growth",
    "arguments": {
        "company": "Apple"
    }
}


###
1. Import TOOLS
2. Create build_tool_prompt()
3. Test one question
4. Create baseline_questions
5. Run all questions
6. Extract predicted tools
7. Calculate accuracy
8. Test variation_questions
###

In [9]:
baseline_questions = [
    {
        "question": "How has Apple's revenue grown?",
        "expected_tool": "get_revenue_growth",
        "company": "Apple"
    },
    {
        "question": "Is Apple's profitability improving?",
        "expected_tool": "get_profit_margin",
        "company": "Apple"
    },
    {
        "question": "How quickly are Apple's assets growing?",
        "expected_tool": "get_asset_growth",
        "company": "Apple"
    },
    {
        "question": "How much of Apple's assets are financed by liabilities?",
        "expected_tool": "get_liability_to_asset_ratio",
        "company": "Apple"
    },
    {
        "question": "How has Apple's operating cash flow performed relative to revenue?",
        "expected_tool": "get_operating_cash_flow_margin",
        "company": "Apple"
    },
    {
        "question": "What are Apple's latest financial results?",
        "expected_tool": "get_latest_summary",
        "company": "Apple"
    },
    {
        "question": "What is Apple's long-term revenue growth rate?",
        "expected_tool": "get_revenue_cagr",
        "company": "Apple"
    },
    {
        "question": "How has Apple's net income grown over the long term?",
        "expected_tool": "get_net_income_cagr",
        "company": "Apple"
    },
    {
        "question": "Is Apple's net income growing faster than its revenue?",
        "expected_tool": "get_revenue_vs_income_growth",
        "company": "Apple"
    },
    {
        "question": "What are the overall financial trends for Apple?",
        "expected_tool": "get_trend_summary",
        "company": "Apple"
    }
]

In [10]:
results = []

for item in baseline_questions:

    prompt = build_tool_prompt(
        item["question"]
    )

    response = llm.generate(prompt)

    results.append(
        {
            "question": item["question"],
            "expected_tool": item["expected_tool"],
            "model_response": response
        }
    )

In [11]:
results_df = pd.DataFrame(results)

results_df

,question,expected_tool,model_response
0,How has Apple's revenue grown?,get_revenue_growth,"{\n ""tool"": ""get_revenue_growth"",\n ""arg..."
1,Is Apple's profitability improving?,get_profit_margin,"{\n ""tool"": ""get_profit_margin"",\n ""argu..."
2,How quickly are Apple's assets growing?,get_asset_growth,"{\n ""tool"": ""get_asset_growth"",\n ""argum..."
3,How much of Apple's assets are financed by lia...,get_liability_to_asset_ratio,"{\n ""tool"": ""get_liability_to_asset_ratio"",..."
4,How has Apple's operating cash flow performed ...,get_operating_cash_flow_margin,"{\n ""tool"": ""get_operating_cash_flow_margin..."
5,What are Apple's latest financial results?,get_latest_summary,"{\n ""tool"": ""get_latest_summary"",\n ""arg..."
6,What is Apple's long-term revenue growth rate?,get_revenue_cagr,"{\n ""tool"": ""get_revenue_cagr"",\n ""argum..."
7,How has Apple's net income grown over the long...,get_net_income_cagr,"{\n ""tool"": ""get_revenue_cagr"",\n ""argum..."
8,Is Apple's net income growing faster than its ...,get_revenue_vs_income_growth,"{\n ""tool"": ""get_revenue_cagr"",\n ""argum..."
9,What are the overall financial trends for Apple?,get_trend_summary,"{\n ""tool"": ""get_trend_summary"",\n ""argu..."


In [12]:
def extract_tool(response):

    try:
        parsed = json.loads(response)

        return parsed.get("tool")

    except Exception:
        return None

### calculate baseline accuracy

In [13]:
results_df["predicted_tool"] = (
    results_df["model_response"]
    .apply(extract_tool)
)

In [14]:
results_df[
    [
        "question",
        "expected_tool",
        "predicted_tool"
    ]
]

,question,expected_tool,predicted_tool
0,How has Apple's revenue grown?,get_revenue_growth,get_revenue_growth
1,Is Apple's profitability improving?,get_profit_margin,get_profit_margin
2,How quickly are Apple's assets growing?,get_asset_growth,get_asset_growth
3,How much of Apple's assets are financed by lia...,get_liability_to_asset_ratio,get_liability_to_asset_ratio
4,How has Apple's operating cash flow performed ...,get_operating_cash_flow_margin,get_operating_cash_flow_margin
5,What are Apple's latest financial results?,get_latest_summary,get_latest_summary
6,What is Apple's long-term revenue growth rate?,get_revenue_cagr,get_revenue_cagr
7,How has Apple's net income grown over the long...,get_net_income_cagr,get_revenue_cagr
8,Is Apple's net income growing faster than its ...,get_revenue_vs_income_growth,get_revenue_cagr
9,What are the overall financial trends for Apple?,get_trend_summary,get_trend_summary


In [15]:
results_df["correct"] = (
    results_df["expected_tool"]
    == results_df["predicted_tool"]
)

accuracy = results_df["correct"].mean()
print(
    f"Baseline tool-routing accuracy: "
    f"{accuracy:.2%}"
)

Baseline tool-routing accuracy: 80.00%


### Test different wording


In [16]:
variation_questions = [
    {
        "question": "Has Apple been making more money from sales recently?",
        "expected_tool": "get_revenue_growth"
    },
    {
        "question": "Is Apple becoming more profitable?",
        "expected_tool": "get_profit_margin"
    },
    {
        "question": "Is Apple's balance sheet getting bigger?",
        "expected_tool": "get_asset_growth"
    },
    {
        "question": "How dependent is Apple on liabilities?",
        "expected_tool": "get_liability_to_asset_ratio"
    },
    {
        "question": "How strong is Apple's cash generation compared with sales?",
        "expected_tool": "get_operating_cash_flow_margin"
    }
]

In [17]:
results = []

for item in variation_questions:

    prompt = build_tool_prompt(
        item["question"]
    )

    response = llm.generate(prompt)

    results.append(
        {
            "question": item["question"],
            "expected_tool": item["expected_tool"],
            "model_response": response
        }
    )

In [18]:
results_df = pd.DataFrame(results)

results_df

,question,expected_tool,model_response
0,Has Apple been making more money from sales re...,get_revenue_growth,"{\n ""tool"": ""get_revenue_vs_income_growth"",..."
1,Is Apple becoming more profitable?,get_profit_margin,"{\n ""tool"": ""get_profit_margin"",\n ""argu..."
2,Is Apple's balance sheet getting bigger?,get_asset_growth,"{\n ""tool"": ""get_balance_sheet"",\n ""argu..."
3,How dependent is Apple on liabilities?,get_liability_to_asset_ratio,"{\n ""tool"": ""get_liability_to_asset_ratio"",..."
4,How strong is Apple's cash generation compared...,get_operating_cash_flow_margin,"{\n ""tool"": ""get_cash_generation"",\n ""ar..."


In [19]:
results_df["predicted_tool"] = (
    results_df["model_response"]
    .apply(extract_tool)
)

results_df[
    [
        "question",
        "expected_tool",
        "predicted_tool"
    ]
]

results_df["correct"] = (
    results_df["expected_tool"]
    == results_df["predicted_tool"]
)

accuracy = results_df["correct"].mean()
print(
    f"Baseline tool-routing accuracy2: "
    f"{accuracy:.2%}"
)

Baseline tool-routing accuracy2: 40.00%


### Create our few-shot examples


#### They are simply examples included in the prompt.

In [20]:
few_shot_examples = [
    {
        "question": "How has Apple's revenue grown?",
        "tool": "get_revenue_growth",
    },
    {
        "question": "Is Apple's profitability improving?",
        "tool": "get_profit_margin",
    },
    {
        "question": "How quickly are Apple's assets growing?",
        "tool": "get_asset_growth",
    },
    {
        "question": "How much of Apple's assets are financed by liabilities?",
        "tool": "get_liability_to_asset_ratio",
    },
    {
        "question": "How has Apple's operating cash flow performed relative to revenue?",
        "tool": "get_operating_cash_flow_margin",
    },
    {
        "question": "What are Apple's latest financial results?",
        "tool": "get_latest_summary",
    },
    {
        "question": "What is Apple's long-term revenue growth rate?",
        "tool": "get_revenue_cagr",
    },
    {
        "question": "How has Apple's net income grown over the long term?",
        "tool": "get_net_income_cagr",
    },
    {
        "question": "Is Apple's net income growing faster than its revenue?",
        "tool": "get_revenue_vs_income_growth",
    },
    {
        "question": "What are the overall financial trends for Apple?",
        "tool": "get_trend_summary",
    },
]

#### Build a few-shot prompt

In [21]:
def build_few_shot_prompt(question):

    examples_text = ""

    for example in few_shot_examples:

        examples_text += f"""
Question:
{example["question"]}

Correct tool:
{example["tool"]}

---
"""

    tool_descriptions = []

    for name, tool in TOOLS.items():

        tool_descriptions.append(
            f"""
Tool: {name}

Description:
{tool["description"]}
"""
        )

    tools_text = "\n".join(tool_descriptions)

    prompt = f"""
You are a financial analysis assistant.

Select the single best tool for the user's question.

Available tools:

{tools_text}

Here are examples showing how questions
map to tools:

{examples_text}

Now classify this new question:

Question:
{question}

Return ONLY this format:

{{
    "tool": "tool_name",
    "arguments": {{
        "company": "company_name"
    }}
}}
"""

    return prompt

#### Don't test using the exact examples This is extremely important.

In [22]:
few_shot_test_questions = [
    {
        "question": "Are Apple's assets increasing over time?",
        "expected_tool": "get_asset_growth",
    },
    {
        "question": "Has Apple's profit margin been getting better?",
        "expected_tool": "get_profit_margin",
    },
    {
        "question": "How quickly has Microsoft's net income increased over the years?",
        "expected_tool": "get_net_income_cagr",
    },
    {
        "question": "Is Apple's sales growth keeping up with its earnings growth?",
        "expected_tool": "get_revenue_vs_income_growth",
    },
    {
        "question": "How much operating cash flow does Apple generate compared with its revenue?",
        "expected_tool": "get_operating_cash_flow_margin",
    },
]

#### Run the few-shot experiment

In [23]:
few_shot_results = []

for item in few_shot_test_questions:

    prompt = build_few_shot_prompt(
        item["question"]
    )

    response = llm.generate(prompt)

    predicted_tool = extract_tool(response)

    few_shot_results.append(
        {
            "question": item["question"],
            "expected_tool": item["expected_tool"],
            "predicted_tool": predicted_tool,
            "model_response": response,
        }
    )

In [24]:
few_shot_df = pd.DataFrame(
    few_shot_results
)

few_shot_df[
    [
        "question",
        "expected_tool",
        "predicted_tool",
    ]
]



,question,expected_tool,predicted_tool
0,Are Apple's assets increasing over time?,get_asset_growth,get_asset_growth
1,Has Apple's profit margin been getting better?,get_profit_margin,get_profit_margin
2,How quickly has Microsoft's net income increas...,get_net_income_cagr,get_net_income_cagr
3,Is Apple's sales growth keeping up with its ea...,get_revenue_vs_income_growth,get_revenue_cagr
4,How much operating cash flow does Apple genera...,get_operating_cash_flow_margin,get_operating_cash_flow_margin


In [25]:
few_shot_df["correct"] = (
    few_shot_df["expected_tool"]
    == few_shot_df["predicted_tool"]
)

few_shot_accuracy = (
    few_shot_df["correct"].mean()
)

print(
    f"Few-shot tool-routing accuracy: "
    f"{few_shot_accuracy:.2%}"
)

Few-shot tool-routing accuracy: 80.00%


### Build a proper routing evaluation dataset

#### Evaluate the 50-question dataset

In [26]:
from src.tools.tool_router import ToolRouter

router = ToolRouter()

result = router.route_question(
    "How has Apple's revenue grown?"
)

result

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [27]:
router.route_question(
    "Is Apple's profitability improving?"
)